# 03 Supervision ladder

**Question.** What are the rung-1, rung-2 pooled-OOF, and random-group numbers, and how do per-fold raw vs floor differ?

In [ ]:
from pathlib import Path
import os, json, csv
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "chapter_a").is_dir():
    for cand in (Path(".."), Path("../.."), Path("../../..")):
        if (cand.resolve() / "chapter_a").is_dir():
            ROOT = cand.resolve()
            break
os.chdir(ROOT)
MASTER = pd.read_csv(ROOT / "chapter_a" / "MASTER_RESULTS.csv")
ANDROCT = ROOT / "abrg" / "output" / "androct_2017"

def row_eq(mask, artifact_auc):
    sub = MASTER.loc[mask]
    assert len(sub) >= 1, mask
    mval = float(sub.iloc[0]["auc_floor"])
    aval = float(artifact_auc)
    assert round(mval, 6) == round(aval, 6), (mval, aval)


In [ ]:
print(pd.read_csv(ROOT / "chapter_a" / "tables" / "T4_supervision_ladder.csv").to_string(index=False))
r1 = json.loads((ANDROCT / "ladder" / "rung1" / "rung1.json").read_text())
r2 = json.loads((ANDROCT / "ladder" / "rung2" / "behavioral_group_holdout.json").read_text())
rg = json.loads((ANDROCT / "ladder" / "control" / "random_group_holdout.json").read_text())
hgb = r1["modes"]["full"]["models"]["hist_gradient_boosting"]["auc"]["auc_floor"]
pooled = r2["pooled_oof_hgb_full"]["auc_floor"]
row_eq((MASTER.experiment=="ladder") & (MASTER.detector=="HGB") & (MASTER.method=="supervised"), hgb)
row_eq(MASTER.detector=="HGB_pooled_oof_raw", pooled)
n_inv = sum(1 for f in r2["folds"] if f["modes"]["full"]["hist_gradient_boosting"]["auc"]["auc"] < 0.5)
print("n_folds raw<0.5", n_inv)
print("random-group mean floor", rg.get("aggregate", {}).get("full", {}))
print("ok")
